# Dataset

In [79]:
text = ""
with open("tinyshakesphere.txt") as f:
    text = f.read()

print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [80]:
vocab = list(set(text))
vocab_size = len(vocab)
print(vocab_size)

65


In [81]:
encode = {val:i for i,val in enumerate(vocab)}
decode = {i:val for i,val in enumerate(vocab)}

print(decode[0])

r


In [82]:
import torch
tokenized_text = torch.tensor([encode[c] for c in text],dtype=torch.long)
print(tokenized_text[:1000])

tensor([30,  7,  0, 33, 39, 45, 58,  7, 39,  7, 62, 42, 14, 37,  8, 27, 42, 63,
        21,  0, 42, 45,  4, 42, 45, 55,  0, 21, 54, 42, 42, 26, 45, 28, 14, 57,
        45, 63, 31,  0, 39, 46, 42,  0, 38, 45, 46, 42, 28,  0, 45, 47, 42, 45,
        33, 55, 42, 28, 16,  5,  8,  8,  1, 53, 53, 37,  8, 13, 55, 42, 28, 16,
        38, 45, 33, 55, 42, 28, 16,  5,  8,  8, 30,  7,  0, 33, 39, 45, 58,  7,
        39,  7, 62, 42, 14, 37,  8, 32, 21, 31, 45, 28,  0, 42, 45, 28, 53, 53,
        45,  0, 42, 33, 21, 53, 19, 42, 26, 45,  0, 28, 39, 46, 42,  0, 45, 39,
        21, 45, 26,  7, 42, 45, 39, 46, 28, 14, 45, 39, 21, 45, 63, 28, 47,  7,
        33, 46, 43,  8,  8,  1, 53, 53, 37,  8, 64, 42, 33, 21, 53, 19, 42, 26,
         5, 45,  0, 42, 33, 21, 53, 19, 42, 26,  5,  8,  8, 30,  7,  0, 33, 39,
        45, 58,  7, 39,  7, 62, 42, 14, 37,  8, 30,  7,  0, 33, 39, 38, 45, 57,
        21, 31, 45, 16, 14, 21,  4, 45, 58, 28,  7, 31, 33, 45, 34, 28,  0, 54,
         7, 31, 33, 45,  7, 33, 45, 54, 

In [83]:
train_data = tokenized_text[:int(0.9*len(text))]
val_data = tokenized_text[int(0.9*len(text)):]

print(len(train_data),len(val_data))

1003854 111540


In [84]:
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data

    random_starting_indices = torch.randint(len(data) - block_size, (block_size,))

    x = torch.stack([data[i:i+block_size] for i in random_starting_indices])
    y = torch.stack([data[i+1:i+block_size+1] for i in random_starting_indices])
    return x,y
    

x,y = get_batch("train")
print(x)
print(y)


tensor([[21, 53, 16, 33, 38,  8, 46, 21],
        [ 3,  6, 41, 11, 32, 58,  3, 13],
        [45, 26,  7, 19, 42,  0, 33, 45],
        [ 0, 45, 33, 28, 54,  0, 42, 26],
        [45, 29,  0,  7, 14, 50, 45, 47],
        [28, 39,  7, 21, 14, 45, 63, 21],
        [ 8,  1, 14, 26, 45, 29,  0, 42],
        [33,  4, 42, 28,  0, 45, 39, 21]])
tensor([[53, 16, 33, 38,  8, 46, 21,  4],
        [ 6, 41, 11, 32, 58,  3, 13, 37],
        [26,  7, 19, 42,  0, 33, 45, 26],
        [45, 33, 28, 54,  0, 42, 26, 45],
        [29,  0,  7, 14, 50, 45, 47, 42],
        [39,  7, 21, 14, 45, 63, 21,  0],
        [ 1, 14, 26, 45, 29,  0, 42, 28],
        [ 4, 42, 28,  0, 45, 39, 21,  8]])


# Bigram Model

In [105]:
import torch.nn as nn
import torch.nn.functional as F

class BigramModel(nn.Module):
    def __init__(self,n_embeddings, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=n_embeddings, embedding_dim=vocab_size)

    def forward(self,input_encodings,targets_encodings=None):
        input_embeddings = self.embedding(input_encodings) # (B,T,C) , B = block_size, T = length of sequence, C = embedding_dim

        loss = None
        if targets_encodings is not None:
            input_embeddings_for_loss = input_embeddings.permute(0,2,1) # (B,C,T)
            loss = F.cross_entropy(input_embeddings_for_loss,targets_encodings)
        
        return loss, input_embeddings 
    
    def generate(self,input_encoding, max_output_len):
        for _ in range(max_output_len):
            _,logits = self(input_encoding)
            logits = logits[:,-1,:] # Last char of sequence
            predictions_probs = F.softmax(logits,dim=-1)
            prediction = torch.multinomial(predictions_probs, num_samples=1)
            input_encoding = torch.cat((input_encoding,prediction),dim=1)

        return input_encoding
    
model = BigramModel(vocab_size, vocab_size) # 1 row for each possible char, and for softmax, we need the vocab as output

start_context = torch.zeros((1, 1), dtype=torch.long)
gen = model.generate(start_context, 100)
print(''.join([decode[int(i)] for i in gen[0]]))



rFvsXYLADa'w?-PjisSgppHFH&Ds
;d&FYZE?u.'VMrtn
QE???D!Ge$MaSLi&D!VMENKlbA&wSZLZGr;
dd3iUk!p:KEWWs
'G;&


# Training Bigram

In [107]:
device = "cuda" if torch.cuda.is_available else 'cpu'
print(device)

model = model.to(device)

cuda


In [ ]:
def train(model, n_steps, optimizer, device):
    model.train()
    
    for step in range(n_steps):
        x, y = get_batch('train')
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()       
        loss, _ = model(x, y)   
        loss.backward()         
        optimizer.step()            
        
        if step % 100 == 0:
            print(f"Step {step} | Train Loss = {loss.item():.4f}")


import torch.optim as optim

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
train(model, 4000, optimizer,device)

Step 0 | Train Loss = 2.8282
Step 100 | Train Loss = 2.5762
Step 200 | Train Loss = 2.7935
Step 300 | Train Loss = 2.6517
Step 400 | Train Loss = 2.6060
Step 500 | Train Loss = 2.9973
Step 600 | Train Loss = 2.8471
Step 700 | Train Loss = 2.7266
Step 800 | Train Loss = 2.5791
Step 900 | Train Loss = 2.5840
Step 1000 | Train Loss = 2.5667
Step 1100 | Train Loss = 2.6519
Step 1200 | Train Loss = 2.7184
Step 1300 | Train Loss = 2.6683
Step 1400 | Train Loss = 2.5919
Step 1500 | Train Loss = 2.5063
Step 1600 | Train Loss = 2.8883
Step 1700 | Train Loss = 2.5241
Step 1800 | Train Loss = 2.3548
Step 1900 | Train Loss = 2.7080
Step 2000 | Train Loss = 2.4123
Step 2100 | Train Loss = 2.8351
Step 2200 | Train Loss = 2.4626
Step 2300 | Train Loss = 2.6865
Step 2400 | Train Loss = 2.5565
Step 2500 | Train Loss = 2.5958
Step 2600 | Train Loss = 2.7467
Step 2700 | Train Loss = 2.6350
Step 2800 | Train Loss = 2.4901
Step 2900 | Train Loss = 2.5980
Step 3000 | Train Loss = 2.5791
Step 3100 | Train Lo

In [115]:
def evaluate(model,device):
    model.eval()
    x,y = get_batch('eval')
    x, y = x.to(device), y.to(device)

    loss,_ = model(x,y)
    print("Validation Loss = ",loss.item())

evaluate(model,device)
start_context = torch.zeros((1, 1), dtype=torch.long).to(device)
gen = model.generate(start_context, 100)
print(''.join([decode[int(i)] for i in gen[0]]))


Validation Loss =  2.507030725479126
rche m n ce ney w y us pored d'she--ber n,
D s.
LURoutu s vity I uceth m
creres y, wlivelil.

BAnt.
A
